# ChemBench Tutorial 1: Load and Explore Data

Welcome to ChemBench — a standardized machine learning benchmark suite for chemical engineering.

ChemBench ships **processed, standardized CSV files** under `chembench/datasets/<name>/processed/`. Each dataset follows consistent conventions: molecular benchmarks expose a `smiles` column and task-specific target columns; process datasets (e.g. Tennessee Eastman) use numeric sensor features; mixture datasets (CheMixHub) use task-specific files. The `ChemBenchDataLoader` resolves paths, detects dataset type, and exposes `load_data()`, `get_features()`, and `get_targets()` so you can focus on modeling instead of ad-hoc file handling.

In this notebook you will load the **ESOL** aqueous solubility dataset and apply a **scaffold split** — the recommended evaluation protocol for molecular property prediction.

In [ ]:
from chembench.data.loader import ChemBenchDataLoader
from chembench.data.splitter import DataSplitter

In [ ]:
loader = ChemBenchDataLoader("esol")
df = loader.load_data()

print(f"Dataset shape: {df.shape}")
print(f"Target column: {loader.get_targets().columns[0]}")
df.head()

## Scaffold splitting for molecular data

Random train/test splits can **leak structural information** when similar scaffolds appear in both sets — models may memorize substructures rather than generalize. ChemBench's `DataSplitter.scaffold_split()` groups molecules by their **Bemis–Murcko scaffold** (via RDKit), then assigns whole scaffold groups to train, validation, or test partitions. This mimics realistic deployment where novel core structures must still be predicted accurately.

**Requirements:** RDKit (`pip install rdkit`). Default split sizes are 70% train / 10% validation / 20% test.

In [ ]:
splitter = DataSplitter()
smiles_col = "smiles" if "smiles" in df.columns else "mol"

train_df, val_df, test_df = splitter.scaffold_split(
    df,
    smiles_col=smiles_col,
    test_size=0.2,
    val_size=0.1,
)

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")